# Chapter 2: Waging War — Gateway Cost Discipline

Paired with Chapter 2. Demonstrates the three levers — context, caching, fallback — with the LiteLLM config shape from the book.

No API keys required. The gateway stub is deterministic. Real LiteLLM usage appears in the last cell, commented out.

In [ ]:
import sys, subprocess
result = subprocess.run([sys.executable, '../chapters/02-waging-war/run-eval.py'], capture_output=True, text=True)
print(result.stdout)

## 1. The gateway config
Policy lives here, not in application code. Swap the provider, the cache, the budget, without touching a caller.

In [ ]:
import yaml
with open('../chapters/02-waging-war/guardrail-config.yaml') as f:
    gw = yaml.safe_load(f)
print(yaml.dump(gw, sort_keys=False))

## 2. The anti-pattern
Same traffic, no gateway. The knowledge base prepended to every request, a timestamp in the system prefix, and full history replayed on each turn.

In [ ]:
print(subprocess.run([sys.executable, '../chapters/02-waging-war/anti-pattern-demo.py'], capture_output=True, text=True).stdout)

## 3. The trace
One call through the gateway. Cost, cache status, route, fallback trail — all in one structured object.

In [ ]:
import json
with open('../chapters/02-waging-war/trace-example.json') as f:
    print(json.dumps(json.load(f), indent=2))

## 4. Wire up real LiteLLM (optional)

```python
import litellm, os
os.environ['ANTHROPIC_API_KEY'] = '...'

response = litellm.completion(
    model='anthropic/claude-opus-4-7',
    messages=[
        {'role': 'system', 'content': 'You are a support agent.'},
        {'role': 'user', 'content': 'Why did my order fail?'},
    ],
    fallbacks=[{'model': 'bedrock/anthropic.claude-sonnet-4-7'}],
    metadata={'cache_key': 'support:v1', 'user_id': 'user_42'},
)
print(response.choices[0].message.content)
print('cost_usd:', response._hidden_params.get('response_cost'))
```